# Day 09. Exercise 02
# Metrics

## 0. Imports

In [61]:
import pandas as pd
from sklearn.model_selection import train_test_split

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import RandomForestClassifier

import joblib
from sklearn.metrics import confusion_matrix

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [62]:
df_not_scaled = pd.read_csv('../data/day-of-week-not-scaled.csv')
df_target = pd.read_csv('../data/dayofweek.csv')

df_not_scaled['dayofweek'] = df_target['dayofweek']

In [63]:
X = df_not_scaled.drop('dayofweek', axis=1)
y = df_not_scaled['dayofweek']

In [64]:
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=21, 
    stratify=y
)

## 2. SVM

1. Use the best parameters from the previous exercise and train the model of SVM.
2. You need to calculate `accuracy`, `precision`, `recall`, `ROC AUC`.

 - `precision` and `recall` should be calculated for each class (use `average='weighted'`)
 - `ROC AUC` should be calculated for each class against any other class (all possible pairwise combinations) and then weighted average should be applied for the final metric
 - the code in the cell should display the result as below:

```
accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.97878
```

In [65]:
best_svc = SVC(
    C=10,
    class_weight=None,
    gamma='auto', 
    kernel='rbf', 
    probability=True, 
    random_state=21
)
best_svc.fit(X_train, y_train)

y_pred = best_svc.predict(X_test)
y_score = best_svc.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)

prec = precision_score(y_test, y_pred, average='weighted')
rec = recall_score(y_test, y_pred, average='weighted')

roc_auc = roc_auc_score(y_test, y_score, multi_class='ovo', average='weighted')

print(f"accuracy is {acc:.5f}")
print(f"precision is {prec:.5f}")
print(f"recall is {rec:.5f}")
print(f"roc_auc is {roc_auc:.5f}")

accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.97878


## 3. Decision tree

1. The same task for decision tree

In [ ]:
best_dt = DecisionTreeClassifier(
    max_depth=21, 
    criterion='gini',
    class_weight='balanced',
    random_state=21
)
best_dt.fit(X_train, y_train)

y_pred = best_dt.predict(X_test)
y_score = best_dt.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='weighted')
rec = recall_score(y_test, y_pred, average='weighted')

roc_auc = roc_auc_score(y_test, y_score, multi_class='ovo', average='weighted')

print(f"accuracy is {acc:.5f}")
print(f"precision is {prec:.5f}")
print(f"recall is {rec:.5f}")
print(f"roc_auc is {roc_auc:.5f}")

accuracy is 0.89053
precision is 0.89402
recall is 0.89053
roc_auc is 0.93793


## 4. Random forest

1. The same task for random forest.

In [ ]:
best_rf = RandomForestClassifier(
    class_weight=None,
    n_estimators=100, 
    max_depth=24,
    criterion='entropy',
    random_state=21,
    n_jobs=-1
)

best_rf.fit(X_train, y_train)

y_pred = best_rf.predict(X_test)
y_score = best_rf.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='weighted')
rec = recall_score(y_test, y_pred, average='weighted')

roc_auc = roc_auc_score(y_test, y_score, multi_class='ovo', average='weighted')

print(f"accuracy is {acc:.5f}")
print(f"precision is {prec:.5f}")
print(f"recall is {rec:.5f}")
print(f"roc_auc is {roc_auc:.5f}")

accuracy is 0.93491
precision is 0.93751
recall is 0.93491
roc_auc is 0.98952


## 5. Predictions

1. Choose the best model.
2. Analyze: for which `weekday` your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which `labname` and for which `users`.
3. Save the model.

In [68]:
best_rf = RandomForestClassifier(
    class_weight=None,
    n_estimators=100, 
    max_depth=24, 
    criterion='entropy',
    random_state=21,
    n_jobs=-1
)
best_rf.fit(X_train, y_train)

all_preds = best_rf.predict(X)

### df for errors analyzing

In [76]:
df_analysis = pd.read_csv('../data/day-of-week-not-scaled.csv')
df_target = pd.read_csv('../data/dayofweek.csv')
df_analysis['dayofweek'] = df_target['dayofweek']
df_analysis['predicted_day'] = all_preds
df_analysis['is_error'] = df_analysis['dayofweek'] != df_analysis['predicted_day']

def calculate_error_rate(df, column_name):
    stats = df.groupby(column_name)['is_error'].agg(['count', 'sum'])

    stats['error_rate_%'] = (stats['sum'] / stats['count']) * 100

    return stats.sort_values('error_rate_%', ascending=False)

df_analysis['uid'] = df_analysis.filter(like='uid_').idxmax(axis=1).str.replace('uid_', '')
df_analysis['labname'] = df_analysis.filter(like='labname_').idxmax(axis=1).str.replace('labname_', '')

In [70]:
days_map = {0:'Mon', 1:'Tue', 2:'Wed', 3:'Thu', 4:'Fri', 5:'Sat', 6:'Sun'}
errors_day = calculate_error_rate(df_analysis, 'dayofweek')
errors_day


,count,sum,error_rate_%
dayofweek,,,
0,136,7,5.147059
4,104,3,2.884615
2,149,2,1.342282
5,271,3,1.107011
1,274,3,1.094891
6,356,2,0.561798
3,396,2,0.505051


In [79]:
errors_user = calculate_error_rate(df_analysis, 'uid')
print(errors_user.head(5))

         count  sum  error_rate_%
uid                              
user_6      12    2     16.666667
user_16     32    2      6.250000
user_27     23    1      4.347826
user_18     35    1      2.857143
user_3      71    2      2.816901


In [80]:
print(calculate_error_rate(df_analysis, 'labname').head(5))

         count  sum  error_rate_%
labname                          
lab03        1    1    100.000000
lab03s       1    1    100.000000
laba04     178    6      3.370787
lab05s      36    1      2.777778
laba06      48    1      2.083333


In [82]:
joblib.dump(best_rf, '../data/model_best_rf.joblib')

['../data/model_best_rf.joblib']

## 6. Function

1. Write a function that takes a list of different models and a corresponding list of parameters (dicts) and returns a dict that contains all the 4 metrics for each model.

In [83]:
def evaluate_models(models, params, X_train, y_train, X_test, y_test):
    results = {}

    for ModelClass, param in zip(models, params):
        model = ModelClass(**param)

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)

        metrics = {
            'accuracy': accuracy_score(y_test, y_pred),
            'precision': precision_score(y_test, y_pred, average='weighted'),
            'recall': recall_score(y_test, y_pred, average='weighted'),
            'roc_auc': roc_auc_score(y_test, y_prob, multi_class='ovo', average='weighted')
        }

        results[ModelClass.__name__] = metrics
    
    return results

In [ ]:
models_list = [SVC, DecisionTreeClassifier, RandomForestClassifier]
params_list = [
    # SVC
    {'C': 10, 'gamma': 'auto', 'kernel': 'rbf', 'class_weight': None, 'probability': True, 'random_state': 21},
    
    # Decision Tree
    {'max_depth': 21, 'class_weight': 'balanced', 'criterion': 'gini', 'random_state': 21},
    
    # Random Forest
    {'n_estimators': 100, 'max_depth': 24, 'criterion': 'entropy', 'class_weight': None, 'random_state': 21, 'n_jobs': -1}
]
results_dict = evaluate_models(models_list, params_list, X_train, y_train, X_test, y_test)
print(pd.DataFrame(results_dict).T)

                        accuracy  precision    recall   roc_auc
SVC                     0.887574   0.892673  0.887574  0.978779
DecisionTreeClassifier  0.890533   0.894016  0.890533  0.937929
RandomForestClassifier  0.934911   0.937509  0.934911  0.989522
